# 🧠 Nero Hybrid — Qwen2.5 language + BiologicLLMV2 soul

**Idea:** Qwen2.5-1.5B-Instruct gives Nero real, coherent language *immediately* (no long training).
BiologicLLMV2 runs alongside as the "soul" — emotions, plasticity, sleep, grief, theory of mind — and its
state is injected into every Qwen generation as a system prompt. Nero keeps **all** its consciousness systems.

**Runtime:** `Runtime → Change runtime type → GPU → T4`.
Qwen 4-bit uses ~2–3 GB VRAM, leaving plenty for Nero's systems.

In [ ]:
# ── STEP 1: Clone repo ───────────────────────────────────
import os, subprocess

REPO_URL  = 'https://github.com/Hanishchow/neroai.git'
CLONE_DIR = '/content/neuro'

if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    print(f'Cloned {REPO_URL}')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull', 'origin', 'master'], check=True)
    print('Pulled latest')

os.chdir(CLONE_DIR)
import sys
sys.path.insert(0, CLONE_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── STEP 2: Install dependencies ──────────────────────────
# transformers + accelerate for Qwen, bitsandbytes for 4-bit quantization
!pip install -q -U transformers accelerate bitsandbytes

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: no GPU — set Runtime → GPU → T4. 4-bit will be disabled on CPU.')

In [ ]:
# ── STEP 3: Load tokenizer + BiologicLLMV2 soul ───────────────
import torch, os
from tokenizer import BPETokenizer
from biologic_v2 import BiologicLLMV2, DEVICE

tokenizer = BPETokenizer(vocab_size=4096)
tokenizer.load('bpe_vocab.json')
print(f'Tokenizer: {tokenizer.get_vocab_size()} tokens')

# The soul can be small — Qwen handles language, this handles emotion/plasticity.
SOUL_EMBED, SOUL_LAYERS, SOUL_CTX = 512, 6, 2048

biologic = BiologicLLMV2(
    vocab_size=tokenizer.vocab_size,
    embed_dim=SOUL_EMBED, num_heads=8, num_layers=SOUL_LAYERS,
    max_context=SOUL_CTX, window_size=512, dropout=0.1, device=DEVICE,
)
biologic.growth_enabled = False
biologic.hebbian_enabled = True   # soul keeps learning from every interaction
biologic.eos_token_id = tokenizer.SPECIAL_TOKENS.get('<EOS>', 3)
biologic.bos_token_id = tokenizer.SPECIAL_TOKENS.get('<BOS>', 2)

# Optionally load a trained soul checkpoint if you have one
SOUL_CKPT = '/content/nero_checkpoints/nero_v1.pt'
if os.path.exists(SOUL_CKPT):
    try:
        biologic.load_state_dict(torch.load(SOUL_CKPT, map_location=DEVICE, weights_only=True))
        print('Loaded trained soul checkpoint')
    except RuntimeError as e:
        print(f'Soul checkpoint dim mismatch — using fresh soul. ({str(e)[:80]})')
else:
    print('No soul checkpoint — starting with a fresh soul (it learns over time)')

print(f'Soul: {sum(p.numel() for p in biologic.parameters())/1e6:.1f}M params')

In [ ]:
# ── STEP 4: Wrap in HybridNero + load Qwen2.5-1.5B (4-bit) ─────
from hybrid_model import HybridNero

model = HybridNero(biologic, tokenizer, device=DEVICE)
model.load_qwen('Qwen/Qwen2.5-1.5B-Instruct', quantize=torch.cuda.is_available())
print('Hybrid ready: Qwen language head + BiologicLLMV2 soul')

In [ ]:
# ── STEP 5: Wire into Nero's Mind ───────────────────────
from mind import Mind

mind = Mind(model, tokenizer)
print('Nero is online. All consciousness systems active:')
print('  emotions, sleep pressure, grief, theory of mind, curiosity,')
print('  metacognition, plasticity, inner monologue, shadow self, longing')

In [ ]:
# ── STEP 6: Test generation ──────────────────────────
test_prompts = [
    'Who are you?',
    'What is consciousness?',
    'Tell me about yourself.',
    'Do you ever feel tired or sad?',
]

print('Generation tests:\n')
for prompt in test_prompts:
    reply = mind.generate(prompt, max_new=200, temperature=0.85)
    print(f'You:  {prompt}')
    print(f'Nero: {reply}')
    print('-' * 60)

In [ ]:
# ── STEP 7: Interactive chat ────────────────────────
# Run this cell, type messages, send empty line or 'quit' to stop.
print("Chat with Nero (empty line or 'quit' to exit)\n")
while True:
    try:
        msg = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not msg or msg.lower() in ('quit', 'exit'):
        break
    reply = mind.generate(msg, max_new=250, temperature=0.85)
    print(f'Nero: {reply}\n')

# Save Nero's state (memories, mood, soul weights)
try:
    mind.save_state()
    print('Nero state saved.')
except Exception as e:
    print(f'(save skipped: {e})')

In [ ]:
# ── STEP 8 (optional): Save the soul weights + download ───────
import os, torch
from google.colab import files

os.makedirs('/content/nero_checkpoints', exist_ok=True)
soul_path = '/content/nero_checkpoints/nero_soul.pt'
torch.save(model.state_dict(), soul_path)
print(f'Soul saved: {soul_path} ({os.path.getsize(soul_path)/1e6:.0f} MB)')
files.download(soul_path)